# MARS — ML Eğitim Defteri (Colab)

Bu defter MARS'ın denetimli anomali sınıflandırıcısını eğitir ve `mars_ml.joblib` üretir.

**Etiket şeması** (zemin gerçeği olmadığından kontrollü sentetik senaryolar):
- `pump`, `subtle` → 1 (şüpheli: habersiz/sığ tahta hareketi)
- `news_legit` → 0 (meşru haber hareketi)
- normal günler → 0

**Gerçek vaka ile zenginleştirme:** SPK kararlarından derlediğiniz etiketli günleri
`cases.csv` (sütunlar: `ticker,tarih,etiket`) olarak yükleyip aşağıdaki hücrede
veri setine ekleyebilirsiniz. Sistem uydurma etiket kullanmaz; sizin sağladığınız
etiketler sizin sorumluluğunuzdadır.

In [ ]:
!git clone --depth 1 https://github.com/mehmetefeaytas/mars.git /content/mars
%cd /content/mars
!pip install -q numpy pandas scipy scikit-learn pyarrow requests joblib tabulate

In [ ]:
import sys
sys.path.insert(0, 'src')

from mars import ml
from mars.config import load_settings

settings = load_settings()
print('MARS ML eğitim ortamı hazır.')

## 1. Veri setini üret

40 seed × 3 senaryo × 400 gün. Her senaryoda 8 bilinen enjeksiyon günü vardır.

In [ ]:
X, y = ml.build_dataset(list(range(1, 41)), settings=settings)
print('Örnek sayısı:', len(X), '| pozitif oranı:', round(float(y.mean()), 3))
X.head()

### (Opsiyonel) Gerçek etiketli vakalar

Colab'a `cases.csv` yükleyin (sütunlar: `ticker,tarih,etiket` — etiket 1=şüpheli, 0=meşru).
Dosya varsa, o günlerin sinyalleri `yfinance` üzerinden hesaplanıp veri setine katılır.

In [ ]:
from pathlib import Path

if Path('cases.csv').exists():
    import pandas as pd
    from mars.pipeline import analyze
    cases = pd.read_csv('cases.csv')
    rows, labels = [], []
    for ticker, grp in cases.groupby('ticker'):
        try:
            res = analyze(ticker, settings=settings)
            frame = res.frame
            for _, c in grp.iterrows():
                day = pd.Timestamp(c['tarih'])
                if day in frame.index:
                    rows.append(frame.loc[day, X.columns].to_dict())
                    labels.append(int(c['etiket']))
        except Exception as e:
            print(ticker, 'alınamedı:', e)
    if rows:
        Xc, yc = pd.DataFrame(rows), pd.Series(labels, dtype=int)
        X = pd.concat([X, Xc], ignore_index=True)
        y = pd.concat([pd.Series(y), yc], ignore_index=True)
        print('Gerçek vaka eklendi:', len(rows))
else:
    print('cases.csv yok — yalnızca sentetik senaryolarla eğitiliyor.')

## 2. Modeli eğit

In [ ]:
trained = ml.train(list(range(1, 41)), settings=settings)
print('Holdout AUC:', round(trained.auc, 4))
print('Katsayılar:')
for c, w in trained.artifact['meta']['coefficients'].items():
    print(f'  {c:24s} {w:+.3f}')

## 3. Kaydet ve indir

Üretilen `mars_ml.joblib` dosyasını indirip repodaki `data/models/` klasörüne koyun.
Böylece MARS arayüzü ve pipeline `sig_ml` sinyalini otomatik kullanır.

In [ ]:
path = ml.save_model(trained, settings=settings)
print('Kaydedildi:', path)

try:
    from google.colab import files
    files.download(str(path))
except Exception:
    print('Colab dışında çalışıyor — dosya yerel dizinde.')

## 4. Hızlı doğrulama

Modeli yeni bir sentetik senaryoda (eğitimde kullanılmayan seed) test eder.

In [ ]:
from mars.experiments import run_scenario
from sklearn.metrics import roc_auc_score
import numpy as np

r = run_scenario(seed=999, kind='subtle', settings=settings)
# run_scenario sinyalleri yeniden hesaplar; basitlik icin suphe skorunu modelle karsilastir
sf_frame = r.suspicion  # fused
print('MARS füzyon şüphe skoru (enjeksiyon ort / taban ort):',
      round(float(r.suspicion[r.labels == 1].mean()), 1), '/',
      round(float(r.suspicion[r.labels == 0].mean()), 1))
print('Referans AUC (füzyon):', round(roc_auc_score(r.labels, r.suspicion), 3))